In [1]:
import pandas as pd
import glob
import os
import re

In [2]:
#gather all absolute file paths in raw data folder and place them in a list
def get_files():
    raw_data_path = os.path.abspath(os.path.join(os.getcwd(),".."))
    pattern = os.path.join(raw_data_path,"data","raw","*.xlsx")
    file_lst = glob.glob(pattern)
    return file_lst

In [3]:
#clean and filter data to acceptable parameters
def clean_filter(quarter_df_lst):
    #strip and force lowercase on all, force single "_" between words 
    for q in quarter_df_lst:
        q.columns = q.columns.str.lower()
        q.columns = q.columns.str.strip()
        q.columns = q.columns.str.replace('\\s+', '_', regex=True)
    #filter out any sheets with less than 5 rows
    length_check = [v for v in quarter_df_lst if len(v) > 5]
    #filter out sheets without expected cols
    required_cols = {"start","details","status_text","booking_source","player_count","tee_sheet","date_cancelled"}
    col_check = [v for v in length_check if required_cols.issubset(set(v.columns))]
    #filter out sheets with more than 10% NA values on the start time
    valid_start_time = [v for v in col_check if v["start"].isna().mean() < 0.10]
    final = valid_start_time
    return final
    
#for each file, collect each tab into a dict w/ tab name as key, df of data as value. concat all tabs into a full year df
year_dfs = []
file_lst = get_files()
for file in file_lst:
    quarter_dict = pd.read_excel(file,sheet_name=None)
    quarters = list(quarter_dict.values())
    quarters = clean_filter(quarters)
    if quarters:
        year = pd.concat(quarters)
        year_dfs.append(year)
    else: 
        print(f"'{os.path.basename(file)}': no valid data")

#concat all year dfs into one master df
master_raw = pd.concat(year_dfs,ignore_index=True)


In [4]:
#print(master_raw.dtypes)

start             object
details           object
status_text       object
booking_source    object
player_count       int64
tee_sheet         object
date_cancelled    object
dtype: object


In [8]:
#basic cleaning, convert start to datetime, drop booking_source (provides no meaningful input here), rename start to tee_time for clarity
master_df = master_raw.copy()
master_df["start"] = pd.to_datetime(master_df["start"])
master_df.drop(columns=["booking_source"],inplace=True)
master_df.rename(columns={"start":"tee_time"},inplace=True)

In [40]:
#clean time, date, and cost out of details using regex,drop any rows that extract time data to nan
master_df[["book_hr","book_min","am_pm","book_month","book_day","cost_per_group"]] = master_df["details"].str.extract(r"@\s(\d{1,2}):(\d{2})(am|pm|AM|PM|Am|Pm)\s(\d{1,2})\/(\d{1,2})(?:.*?\$(\d+\.\d{2}))?")
master_df.dropna(subset=["book_hr","book_min","am_pm","book_month","book_day"],inplace=True)
master_df[["book_hr","book_min","book_month","book_day"]] = master_df[["book_hr","book_min","book_month","book_day"]].astype(int)

#reformat book time to 24hr to enable cleaning
master_df.loc[(master_df["am_pm"] == "pm")&(master_df["book_hr"] != 12), "book_hr"] = master_df.loc[(master_df["am_pm"] == "pm")&(master_df["book_hr"] != 12), "book_hr"]+12
master_df.loc[(master_df["book_hr"] == 12) & (master_df["am_pm"] == "am"), "book_hr"] = 0

# pull year from start, adjust booking time to previous year if booking in dec and tee time in jan
master_df["book_yr"] = master_df["tee_time"].dt.year
master_df.loc[(master_df["tee_time"].dt.month == 1)&(master_df["book_month"] == 12),"book_yr"] = master_df.loc[(master_df["tee_time"].dt.month == 1)&(master_df["book_month"] == 12),"book_yr"]-1

#convert cleaned data to datetime
master_df["booking_time"] = pd.to_datetime(
    {
        "year": master_df["book_yr"],
        "month": master_df["book_month"],
        "day": master_df["book_day"],
        "hour": master_df["book_hr"],
        "minute": master_df["book_min"],
    },
    errors="coerce",
)
# #drop helper date columns to clean up df, reorganize cols
master_df.drop(columns=["book_hr","book_min","am_pm","book_month","book_day", "book_yr"],inplace=True)
master_df = master_df[['tee_time', 'details', 'booking_time', 'status_text', 'player_count', 'tee_sheet',
       'date_cancelled', 'cost_per_group']]

#drop any rows where booking time occurs after tee time. No clear explanation for this upon analysis so it will be treated as noise (~1 row impacted)
bad_time_order = master_df.index[master_df["booking_time"] > master_df["tee_time"]]
master_df.drop(bad_time_order,inplace=True)


In [41]:
# #investigate tee time before booking time discrepancy
# book_after_tee_time = master_df.loc[master_df["booking_time"] > master_df["tee_time"]]
# display(book_after_tee_time)

,tee_time,details,booking_time,status_text,player_count,tee_sheet,date_cancelled,cost_per_group


In [42]:
#evaluating scale of bad data to see if it makes sense to delete those records, or apply some further...
#cleaning to allow more numerical analysis
for col in master_df.columns:
    display(master_df[col].value_counts(dropna=False))
    display(master_df.loc[master_df[col].isna()])

tee_time
2024-06-03 06:39:00    47
2024-06-15 17:00:00    45
2024-06-10 06:03:00    40
2024-05-23 07:06:00    39
2024-04-13 08:27:00    39
                       ..
2025-10-21 09:21:00     1
2025-10-21 09:30:00     1
2025-10-21 12:30:00     1
2025-10-21 12:48:00     1
2025-10-21 12:57:00     1
Name: count, Length: 117814, dtype: int64

,tee_time,details,booking_time,status_text,player_count,tee_sheet,date_cancelled,cost_per_group


details
Reserved using online booking @ 7:00pm 6/27 EDT Original amount due at course: 4 Players for $192.00                                 266
Reserved using online booking @ 7:00pm 5/30 EDT Original amount due at course: 4 Players for $172.00                                 238
Reserved using online booking @ 7:00pm 6/21 EDT Original amount due at course: 4 Players for $172.00                                 235
Reserved using online booking @ 7:00pm 6/20 EDT Original amount due at course: 4 Players for $172.00                                 228
Reserved using online booking @ 7:00pm 6/12 EDT Original amount due at course: 4 Players for $192.00                                 226
                                                                                                                                    ... 
Reserved using online booking @ 7:41pm 10/14 EDT Original amount due at course: 2 Players for $86.00 - PAID BOOKING FEE OF $10.00      1
Reserved using online booking @ 8

,tee_time,details,booking_time,status_text,player_count,tee_sheet,date_cancelled,cost_per_group


booking_time
2022-05-21 19:00:00    233
2022-05-22 19:00:00    218
2022-07-10 19:00:00    216
2022-06-26 19:00:00    214
2023-06-09 19:00:00    213
                      ... 
2021-03-13 22:55:00      1
2021-03-13 23:18:00      1
2021-03-13 20:26:00      1
2021-03-13 20:37:00      1
2021-03-13 18:42:00      1
Name: count, Length: 374337, dtype: int64

...

,tee_time,details,booking_time,status_text,player_count,tee_sheet,date_cancelled,cost_per_group


date_cancelled
NaT                           250431
NaN                            57373
September 24, 2021 7:31 AM        18
July 09, 2021 4:36 AM             18
2022-03-28 18:43:09               18
                               ...  
2021-03-14 13:07:59                1
2021-03-15 10:32:37                1
2021-03-11 15:10:08                1
2021-03-15 10:32:20                1
2021-03-13 13:39:15                1
Name: count, Length: 268141, dtype: int64

,tee_time,details,booking_time,status_text,player_count,tee_sheet,date_cancelled,cost_per_group
0,2021-03-14 07:51:00,Reserved using online booking @ 7:00pm 3/7 EST...,2021-03-07 19:00:00,checked in,2,Bethpage Early AM 9 Holes Blue,NaT,124.00
1,2021-03-14 07:51:00,Reserved using online booking @ 9:35pm 3/13 ES...,2021-03-13 21:35:00,checked in,1,Bethpage Early AM 9 Holes Blue,NaT,31.00
2,2021-03-14 08:00:00,Reserved using online booking @ 3:32pm 3/13 ES...,2021-03-13 15:32:00,checked in,2,Bethpage Early AM 9 Holes Blue,NaT,62.00
3,2021-03-14 08:27:00,Reserved using online booking @ 11:19am 3/13 E...,2021-03-13 11:19:00,checked in,4,Bethpage Early AM 9 Holes Blue,NaT,124.00
4,2021-03-14 08:36:00,Reserved using online booking @ 5:42pm 3/13 ES...,2021-03-13 17:42:00,checked in,1,Bethpage Early AM 9 Holes Blue,NaT,93.00
...,...,...,...,...,...,...,...,...
616026,2025-10-21 13:33:00,Reserved using online booking @ 7:41pm 10/14 E...,2025-10-14 19:41:00,NaN,2,Bethpage Red Course,NaT,86.00
616027,2025-10-21 13:42:00,Reserved using online booking @ 8:00pm 10/14 E...,2025-10-14 20:00:00,NaN,1,Bethpage Blue Course,NaT,15.00
616028,2025-10-21 13:42:00,Reserved using online booking @ 12:32pm 10/15 ...,2025-10-15 12:32:00,NaN,1,Bethpage Blue Course,NaT,23.00
616029,2025-10-21 13:51:00,Reserved using online booking @ 7:02pm 10/14 E...,2025-10-14 19:02:00,NaN,4,Bethpage Blue Course,NaT,92.00


cost_per_group
192.00    69824
172.00    67826
43.00     41383
48.00     41335
86.00     31698
          ...  
171.00        1
404.00        1
202.00        1
57.00         1
126.00        1
Name: count, Length: 138, dtype: int64

,tee_time,details,booking_time,status_text,player_count,tee_sheet,date_cancelled,cost_per_group
763,2021-03-23 07:42:00,Reserved using online booking @ 9:31am 3/20 ED...,2021-03-20 09:31:00,checked in,1,Bethpage Blue Course,NaT,NaN
1270,2021-03-25 17:36:00,Reserved using online booking @ 2:57pm 3/23 ED...,2021-03-23 14:57:00,deleted,1,Bethpage Green Course,2021-03-23 13:07:46,NaN
3408,2021-04-06 15:39:00,Reserved using online booking @ 8:17am 4/3 EDT...,2021-04-03 08:17:00,deleted,4,Bethpage Yellow 9 Hole Course,2021-04-03 07:01:59,NaN
3991,2021-04-08 17:09:00,Reserved using online booking @ 8:59am 4/7 EDT...,2021-04-07 08:59:00,checked in,2,Bethpage Yellow 9 Hole Course,NaT,NaN
5056,2021-04-14 09:03:00,Reserved using online booking @ 6:15am 4/13 ED...,2021-04-13 06:15:00,checked in,1,Bethpage Green Course,NaT,NaN
...,...,...,...,...,...,...,...,...
615355,2025-10-13 09:03:00,Reserved using online booking @ 7:10pm 10/6 EDT,2025-10-06 19:10:00,deleted,4,Bethpage Red Course,2025-10-10 17:24:01,NaN
615632,2025-10-16 09:03:00,Reserved using online booking @ 2:44pm 10/15 E...,2025-10-15 14:44:00,checked in,1,Bethpage Early AM 9 Holes Blue,NaT,NaN
615653,2025-10-16 11:45:00,Reserved using online booking @ 7:28pm 10/9 EDT,2025-10-09 19:28:00,NaN,2,Bethpage Red Course,NaT,NaN
615966,2025-10-20 13:06:00,Reserved using online booking @ 10:31pm 10/13 EDT,2025-10-13 22:31:00,deleted,2,Bethpage Blue Course,2025-10-13 22:56:01,NaN


In [44]:
#evaluating proper treatment for status_text column, specifically NAN values
status = master_df[["tee_time","booking_time","status_text","date_cancelled"]].copy()
status["book_lead_time"] = master_df["booking_time"]-master_df["tee_time"]
status_na = status.loc[status["status_text"].isna()]
display(status_na)
 

,tee_time,booking_time,status_text,date_cancelled,book_lead_time
77,2021-03-16 11:00:00,2021-03-14 21:16:00,NaN,NaT,-2 days +10:16:00
78,2021-03-16 12:12:00,2021-03-09 19:00:00,NaN,NaT,-7 days +06:48:00
79,2021-03-16 12:39:00,2021-03-15 11:59:00,NaN,NaT,-2 days +23:20:00
80,2021-03-16 13:15:00,2021-03-09 19:00:00,NaN,NaT,-7 days +05:45:00
81,2021-03-16 13:51:00,2021-03-12 20:32:00,NaN,NaT,-4 days +06:41:00
...,...,...,...,...,...
616026,2025-10-21 13:33:00,2025-10-14 19:41:00,NaN,NaT,-7 days +06:08:00
616027,2025-10-21 13:42:00,2025-10-14 20:00:00,NaN,NaT,-7 days +06:18:00
616028,2025-10-21 13:42:00,2025-10-15 12:32:00,NaN,NaT,-7 days +22:50:00
616029,2025-10-21 13:51:00,2025-10-14 19:02:00,NaN,NaT,-7 days +05:11:00
